# Modelos No Supervisados - Validación en Métodos de Clustering

Objetivo -> Análisis **exploratorio** + Análisis **confirmatorio**

## Análisis Exploratorio:

- Para entender patrones, tendencias, detectar anomalías
- Para resumir visualmente datos
- Para generar hipótesis que luego puedan ser confirmadas

## Análisis Confirmatorio:
Formas de validar/confirmar lo encontrado en el Análisis Exploratorio

- Corroborando con literatura
- Validando resultado mediante datos de test
- Validando la estabilidad de los resultados
- Verificando experimentalmente

## *Validando resultado mediante **datos de test***

Dividir los datos en Training y Test, antes del análisis.

Emplear Training para Exploratorio, y Test para Confirmatorio.

- Encontrar los K-clusters con training
- Luego, encontrar también K-clusters con test
- Con training, construir un modelo predictivo para K-etiquetas
- Predecir las etiquetas con los datos de test
- Comparar k-clusters de test con etiquetas predichas mediante métricas (Rand, Jaccard, Adjusted Rand Index)

***Rand Index (RI)***: mide la proporción de pares de puntos que están asignados igual o diferente en ambas particiones.

***Adjusted Rand Index (ARI)***: corrige el RI para el azar, de modo que un valor de 0 indica asignaciones aleatorias y 1 coincidencia perfecta.

***Jaccard Index***: similar a RI, pero considera solo los pares que están juntos en al menos una de las particiones.

| Métrica                       | Función                               | Qué mide                                                                               | Rango                             |
| ----------------------------- | ------------------------------------- | -------------------------------------------------------------------------------------- | --------------------------------- |
| **Rand Index**                | `rand_score`                          | Proporción de pares de muestras con la misma/diferente asignación en ambas particiones | 0–1                               |
| **Adjusted Rand Index (ARI)** | `adjusted_rand_score`                 | Igual que Rand, pero corrige el efecto del azar                                        | −1 a 1 (≈0 aleatorio, 1 perfecto) |
| **Jaccard Index**             | `jaccard_score(..., average='macro')` | Coincidencia promedio entre etiquetas (parecido a IoU)                                 | 0–1                               |


¿Cuándo usar macro o micro en el hiperparámetro average de Jaccard?

| Tipo de promedio      | Cómo calcula                                                                                                                                | Qué significa                                                         | Cuándo usar                                                                       |
| --------------------- | ------------------------------------------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------- | --------------------------------------------------------------------------------- |
| **`average='macro'`** | Calcula la métrica **por clase** y luego hace un **promedio simple** (todas las clases pesan igual)                                         | Evalúa el rendimiento **medio por clase**, sin importar su tamaño     | Si se busca que **todas las clases/clústeres cuenten igual**, incluso los pequeños |
| **`average='micro'`** | Combina **todas las predicciones** y **cuenta verdaderos positivos, falsos positivos y negativos globalmente** antes de calcular la métrica | Evalúa el rendimiento **global**, dando más peso a las clases grandes | Si interesa el desempeño **ponderado por el tamaño del clúster**               |


In [ ]:
from sklearn.metrics import rand_score, adjusted_rand_score, jaccard_score
import numpy as np

true_labels = np.array([0, 0, 1, 1, 2, 2, 0, 0, 1, 1, 2])
pred_labels = np.array([1, 1, 0, 0, 2, 2, 1, 1, 0, 0, 2])

# Rand Index
ri = rand_score(true_labels, pred_labels)

# Adjusted Rand Index
ari = adjusted_rand_score(true_labels, pred_labels)

# Jaccard Index
ji = jaccard_score(true_labels, pred_labels, average='micro')

print(f"Rand Index: {ri:.3f}")
print(f"Adjusted Rand Index: {ari:.3f}")
print(f"Jaccard Index (micro): {ji:.3f}")

## wd

In [ ]:
### Mounting Google Drive :
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
working_dir = "/content/drive/MyDrive/_PUCP_s/Cursos/1INF03_ADD/datos"
%cd $working_dir

In [ ]:
!pwd

In [ ]:
import pandas as pd

### Librerías y datos : "**datos_ascensoTrabajadores.csv**"

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
df = pd.read_csv('datos_ascensoTrabajadores.csv', sep=';')
print(df.shape)
df.head(5)

In [ ]:
df.columns

In [ ]:
df.dtypes

Cambiamos tipo de variables

In [ ]:
df[['employee_id', 'department', 'region',
    'education', 'gender',
    'recruitment_channel',
    'is_promoted',
    'KPIs_met_>80',
    'awards_won'
    ]] = df[['employee_id', 'department', 'region',
             'education', 'gender',
             'recruitment_channel', 'is_promoted',
             'KPIs_met_>80', 'awards_won'
             ]].astype('object')
df.dtypes

In [ ]:
df.nunique()

In [ ]:
df.index = df['employee_id']
df.drop(['employee_id', 'is_promoted', 'department', 'region', 'awards_won'], axis=1, inplace=True)

df.head(5)

In [ ]:
df.dropna(inplace=True)

nas = pd.DataFrame(df.isnull().sum()).sort_values(0,ascending=False)
nas.columns = ['nas']
nas['nas%'] = round(nas['nas']/df.shape[0], 2)
nas

### Partición de los datos

Particionamos los datos en Entrenamiento y Prueba

In [ ]:
print(df.shape)

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test = train_test_split(df, test_size=0.25, random_state=2025)

print(x_train.shape)
print(x_test.shape)

Identificamos y separamos las variables numéricas

In [ ]:
x_train_numericas = x_train.select_dtypes(include='number')
print(x_train_numericas.shape)

x_test_numericas = x_test.select_dtypes(include='number')
print(x_test_numericas.shape)


x_train_categoricas = x_train.select_dtypes(include=['object', 'string'])
print(x_train_categoricas.shape)

x_test_categoricas = x_test.select_dtypes(include=['object', 'string'])
print(x_test_categoricas.shape)

### Transformaciones

Scaler

In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()

x_train_numericas_scaled = scaler.fit_transform(x_train_numericas)

x_train_numericas_scaled = pd.DataFrame(x_train_numericas_scaled,
                                 columns=x_train_numericas.columns,
                                 index=x_train_numericas.index)


# Guardamos el scaler para después usarlo con test
joblib.dump(scaler, 's7cval_transform_scaler.pkl')

# scaler_loaded = joblib.load('transform_scaler.pkl')
# x_new_num_scaled = scaler_loaded.transform(x_new_num)

Generamos variables dummies con las variables categóricas

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
import pandas as pd
import joblib

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')

ohe.fit(x_train_categoricas)

x_train_ohe = ohe.transform(x_train_categoricas)

cols = ohe.get_feature_names_out(  x_train_categoricas.columns  )
x_train_ohe = pd.DataFrame(x_train_ohe, columns=cols, index=x_train.index)

# Guardamos el ohe para después usarlo con test
joblib.dump(ohe, 's7cval_transform_ohe_encoder.pkl')

# ohe_loaded = joblib.load('ohe_encoder.pkl')
# x_new_ohe = ohe_loaded.transform(new_df)


In [ ]:
x_train_ohe.columns

### Transformaciones a 'x_test' para Predicciones

In [ ]:
# scaler_loaded = joblib.load('transform_scaler.pkl')
x_test_numericas_scaled = scaler.transform(x_test_numericas)

x_test_numericas_scaled = pd.DataFrame(x_test_numericas_scaled,
                                 columns=x_test_numericas.columns,
                                 index=x_test_numericas.index)

print(x_test_numericas.shape)
print(x_test_numericas_scaled.shape)

In [ ]:
# ohe_loaded = joblib.load('ohe_encoder.pkl')
x_test_categoricas_ohe = ohe.transform(x_test_categoricas)

cols = ohe.get_feature_names_out(  x_test_categoricas.columns  )
x_test_categoricas_ohe = pd.DataFrame(x_test_categoricas_ohe, columns=cols, index=x_test.index)

print(x_test_categoricas.shape)
print(x_test_categoricas_ohe.shape)

### Concat

In [ ]:
print(x_train.shape)
print(x_train_numericas_scaled.shape)
print(x_train_categoricas.shape)

x_train_v2 = pd.concat([x_train_numericas_scaled,
                             x_train_categoricas], axis=1)
print(x_train_v2.shape)


x_train_v3 = pd.concat([x_train_numericas_scaled,
                             x_train_ohe], axis=1)
print(x_train_v3.shape)

In [ ]:
print(x_test.shape)
print(x_test_numericas_scaled.shape)
print(x_test_categoricas.shape)

x_test_v2 = pd.concat([x_test_numericas_scaled,
                             x_test_categoricas], axis=1)
print(x_test_v2.shape)


x_test_v3 = pd.concat([x_test_numericas_scaled,
                             x_test_categoricas_ohe], axis=1)
print(x_test_v3.shape)

### Modelo: "K-Prototypes"

In [ ]:
!pip install kmodes
!pip install KPrototypes

In [ ]:
from kmodes.kprototypes import KPrototypes

modelo = KPrototypes(random_state = 2025, init = 'random', n_jobs = -1)

**Identificamos posición de variables categóricas**

In [ ]:
catColumnsPos = [x_train_v2.columns.get_loc(col) for col in list(x_train_v2.select_dtypes(['object', 'string', 'category']).columns)]

print('Categorical columns           : {}'.format(list(x_train_v2.select_dtypes(['object', 'string', 'category']).columns)))
print('Categorical columns position  : {}'.format(catColumnsPos))

**Convertimos a matriz**

In [ ]:
Xmatrix = x_train_v2.to_numpy()
Xmatrix

In [ ]:
Xmatrix[1,:]

#### Selección del "K" óptimo para kprototypes

In [ ]:
import itertools
import numpy as np
from kmodes.kprototypes import KPrototypes
import pandas as pd

K_values = [2, 3, 4, 5, 6, 7, 8, 9, 10]
inits = ['Cao']
n_inits = [5, 10]
max_iters = [50, 100]

def gridsearch_kprototypes(X, catCols, K_values, inits, n_inits, max_iters):
    results = []
    for k, init, n_init, max_iter in itertools.product(K_values, inits, n_inits, max_iters):
        print(f"\nProbando: K={k}, init={init}, n_init={n_init}, max_iter={max_iter}")

        model = KPrototypes( n_clusters=k, init=init,  n_init=n_init, max_iter=max_iter, random_state=2025 )
        labels = model.fit_predict(X, categorical=catCols)
        cost = model.cost_
        print(f" → Costo: {cost:.2f}")

        results.append({ 'K': k, 'init': init, 'n_init': n_init, 'max_iter': max_iter, 'cost': cost })

    df_results = pd.DataFrame(results)
    best = df_results.loc[df_results['cost'].idxmin()]

    print("\n Mejor combinación encontrada:")
    print(best)

    return best, df_results


In [ ]:
#X_sample = Xmatrix.sample(n=500, random_state=2025)   #paraDataFrame

X_sample = Xmatrix[np.random.choice(Xmatrix.shape[0], size=1000, replace=False ), :]

In [ ]:
best, df_results = gridsearch_kprototypes(X_sample, catColumnsPos, K_values, inits, n_inits, max_iters)

In [ ]:
import matplotlib.pyplot as plt

# Agrupamos por K y tomamos el costo mínimo para cada K
elbow_data = df_results.groupby('K')['cost'].min().reset_index()

plt.figure(figsize=(8,5))
plt.plot(elbow_data['K'], elbow_data['cost'], marker='o')
plt.title("Elbow Method para KPrototypes")
plt.xlabel("Número de clusters K")
plt.ylabel("Costo mínimo")
plt.xticks(elbow_data['K'])
plt.grid(True)
plt.show()


In [ ]:
K_values = [4]
inits = ['Cao']
n_inits = [5, 10]
max_iters = [50, 100]

best_withK, df_results_withK = gridsearch_kprototypes(X_sample, catColumnsPos, K_values, inits, n_inits, max_iters)

In [ ]:
modelo = KPrototypes(n_clusters = 4, max_iter= 50, n_init= 5, random_state = 2025, init = 'Cao', n_jobs = -1)
modelo.fit_predict(Xmatrix, categorical = catColumnsPos)

In [ ]:
x_train['Clusters_modelo'] = modelo.labels_
x_train['Clusters_modelo']
print(x_train.shape)
x_train.head(5)

In [ ]:
x_train['Clusters_modelo'].value_counts(normalize=True)*100

In [ ]:
modelo.predict(x_test_v2.to_numpy(), categorical = catColumnsPos)

In [ ]:
predicciones = modelo.predict(x_test_v2.to_numpy(), categorical=catColumnsPos)
predicciones

In [ ]:
x_test['Clusters_modelo'] = predicciones
x_test['Clusters_modelo']
print(x_test.shape)
x_test.head(5)

### Modelo de clasificación

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

modelo2 = KNeighborsClassifier( )

Seleccionamos parámetros con GridSearchCV

In [ ]:
param_dist = {
    "n_neighbors": [4, 8, 12, 16, 20],
    "p": [0, 1, 2, 3]
}

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV

skf = StratifiedKFold(n_splits=5, random_state=2025, shuffle=True)

grid = GridSearchCV( modelo2, param_dist, cv=skf.split(x_train_v3, x_train.Clusters_modelo), n_jobs=-1, verbose=1)

grid.fit(x_train_v3, x_train.Clusters_modelo)

print(grid.best_params_)

In [ ]:
param = grid.best_params_
param

Entrenamos el modelo

In [ ]:
modelo2 = KNeighborsClassifier(**param)

modelo2.fit(x_train_v3, x_train.Clusters_modelo)

In [ ]:
modelo2.score(x_train_v3, x_train.Clusters_modelo)

In [ ]:
y_predict_modelo2 = modelo2.predict(x_test_v3)
y_predict_modelo2

In [ ]:
pd.DataFrame(y_predict_modelo2).value_counts()

In [ ]:
pd.DataFrame(y_predict_modelo2).value_counts(normalize=True)*100

In [ ]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

accuracy_score(x_test.Clusters_modelo, y_predict_modelo2)

### Validation

In [ ]:
from sklearn.model_selection import cross_val_score

scores_res = cross_val_score(modelo2, x_train_v3, x_train.Clusters_modelo, cv=10)
print("Accuracy de los 10 folds:", scores_res)
print("Accuracy Mean: {:.3f} (std: {:.3f})".format(scores_res.mean(), scores_res.std()) )

In [ ]:
# Confusion matrix
print(pd.crosstab(x_test.Clusters_modelo, y_predict_modelo2))

In [ ]:
# Rand Index
ri = rand_score(x_test.Clusters_modelo, y_predict_modelo2)

# Adjusted Rand Index
ari = adjusted_rand_score(x_test.Clusters_modelo, y_predict_modelo2)

# Jaccard Index
ji = jaccard_score(x_test.Clusters_modelo, y_predict_modelo2, average='macro')

print(f"Rand Index: {ri:.3f}")
print(f"Adjusted Rand Index: {ari:.3f}")
print(f"Jaccard Index (micro): {ji:.3f}")

In [ ]:
x_train.columns

In [ ]:
import plotly.express as px

fig = px.scatter_3d( x_train, x='no_of_trainings', y='age', z='length_of_service', color='Clusters_modelo'
)
fig.show()

### Modelo

In [ ]:
df_numericas = df.select_dtypes(include='number')
print(df_numericas.shape)

df_categoricas = df.select_dtypes(include=['object', 'string'])
print(df_categoricas.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib

scalerF = StandardScaler()

df_numericas_scaled = scalerF.fit_transform(df_numericas)

df_numericas_scaled = pd.DataFrame(df_numericas_scaled,
                                 columns=df_numericas.columns,
                                 index=df_numericas.index)


In [ ]:
print(df.shape)
print(df_numericas_scaled.shape)

df_v2 = pd.concat([df_numericas_scaled,
                             df_categoricas], axis=1)
print(df_v2.shape)

In [ ]:
modeloFinal = KPrototypes(n_clusters = 4, max_iter= 50, n_init= 5, random_state = 2025, init = 'Cao', n_jobs = -1)
modeloFinal.fit_predict(df_v2.to_numpy(), categorical = catColumnsPos)

In [ ]:
df['Clusters_modeloFinal'] = modeloFinal.labels_
print(df.shape)
df.head(5)

In [ ]:
df['Clusters_modeloFinal'].value_counts(normalize=True)*100

In [ ]:
x_train['Clusters_modelo'].value_counts(normalize=True)*100

In [ ]:
grouped = df.groupby('Clusters_modeloFinal')[df.columns[1]]
stats_por_cluster = grouped.describe()
stats_por_cluster

In [ ]:
# Métricas por cluster
grouped = df.groupby('Clusters_modeloFinal')

# Iteramos sobre cada columna (excepto la columna de clusters)
for col in df.columns:
    if col == 'Clusters_modeloFinal':
        continue  # saltamos la columna de clusters
    print(f"\n%%%%%--- Estadísticas para la columna: {col} ---%%%%%")

    # Iteramos sobre cada cluster
    for cluster_name, datos_cluster in grouped[col]:
        print(f"\nCluster: {cluster_name}")
        print(datos_cluster.describe())


In [ ]:
import plotly.express as px

fig = px.scatter_3d( df, x='no_of_trainings', y='age', z='length_of_service', color='Clusters_modeloFinal' )
fig.show()

## *Validando la **estabilidad** de los resultados*

*Principio de estabilidad:*

*"Si un hallazgo se repite consistentemente bajo pequeñas variaciones de los datos o del modelo, es más probable que sea real"*

Empleamos el 100% de los datos como training data, y mediante perturbaciones generamos nuevos datos de test.
- Si un patrón o clúster importante aparece en casi todas las repeticiones → es estable → probablemente es un hallazgo real.
- Si aparece solo ocasionalmente → probablemente es espurio o sensible al azar.

Otra forma de validar estabilidad es ralizando perturbaciones al proceso de aprendizaje (directamente a los modelos)

Perturbaciones más comunes:
- Subsampling: tomar una fracción del dataset original (por ejemplo, 80% de los datos al azar).
- Bootstrapping: muestrear (con reemplazo) del conjunto original (como en bagging).
- Agregar ruido: modificar levemente las características numéricas.
- Random corruptions: eliminar o distorsionar valores aleatoriamente.
- Random initializations / random tuning: cambiar las semillas o algunos hiperparámetros del modelo.

In [ ]:
'''
#Ejemplo: Evaluar estabilidad de k-means con perturbaciones (bootstrapping + random init)

import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
import matplotlib.pyplot as plt


# Configuración de la validación
K = 3                     # cantidad de clústeres
n_runs = 50               # cantidad de perturbaciones (repeticiones)
sample_frac = 0.8         # fracción de datos usada en cada bootstrap

# Guardamos los labels obtenidos en cada ejecución
all_labels = np.zeros((n_runs, len(X)))


# Repetimos con perturbaciones
rng = np.random.default_rng(42)
for i in range(n_runs):

    # Bootstrapping: seleccionamos una submuestra del 80% de los datos
    idx = rng.choice(len(X), size=int(sample_frac * len(X)), replace=False)
    X_boot = X[idx]

    # Entrenamos K-Means con inicialización aleatoria
    kmeans = KMeans(n_clusters=K, n_init=1, init='random', random_state=None)
    kmeans.fit(X_boot)

    # Predecimos clústeres (para comparar entre runs)
    labels = kmeans.predict(X)  # es una proyección geométrica, no es una predicción supervisada
    all_labels[i, :] = labels


# Medir estabilidad
# Calculamos el Adjusted Rand Index (ARI) entre todas las combinaciones de runs
stabilities = []
for i in range(n_runs):
    for j in range(i+1, n_runs):
        ari = adjusted_rand_score(all_labels[i, :], all_labels[j, :])
        stabilities.append(ari)

mean_stability = np.mean(stabilities)
print(f"Stability (mean Adjusted Rand Index): {mean_stability:.3f}")

# Visualización
plt.hist(stabilities, bins=15, color='skyblue', edgecolor='black')
plt.xlabel("Adjusted Rand Index entre runs")
plt.ylabel("Frecuencia")
plt.title("Distribución de la estabilidad de clustering")
plt.show()
'''